# H&M M4-C 단독 — 마지막 주 seed49
기존 H&M 최종 실행의 빈 비교 한 칸만 채웁니다. **새 학습 대상은 M4-C seed49 한 개뿐**입니다.
- 학습: 2018-09-20~2020-09-15 / test: 2020-09-16~22. 고정 300 epoch, validation·holdout·조기종료 없음.
- CLV 관찰창 365일, 고객 내 평균 1의 M4-C 가중치(계수 0.5), binary 그래프. 기존 설계 변경 없음.
- M1·M3·M5-C는 같은 실행·seed49의 완료 결과만 읽습니다. 없어도 학습하지 않습니다.
- M4-C 완료 결과가 있으면 준비·학습·재평가를 건너뜁니다. 중단됐으면 같은 체크포인트에서 재개합니다.
- **새 L4 런타임**에서 위부터 실행하세요. 기존 M3·M5-C 학습 런타임을 재시작하지 마세요.
- 단일 시드 부분 결과이며 최종 성공 판정·설정 재조정에 사용하지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
# 기존 최종 실행과 같은 전체 커밋: 변경하면 재사용 신원이 달라집니다.
SOURCE_COMMIT = 'c62e861395c128a7207322b70a7e53f9364e8caf'
REPO = Path('/content/clv-hm-m4c-s49-' + SOURCE_COMMIT[:7])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
import clv_m5_user_centered_hm2y_final10 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '새 런타임에서 실행하세요: 다른 코드가 이미 로드됐습니다.'
assert screen.fixed.moe.source_revision() == SOURCE_COMMIT, '실행 소스가 수정됐습니다. 임의로 재사용하지 않습니다.'
print('고정 코드:', SOURCE_COMMIT)


In [ ]:
from dataclasses import asdict
import math

cfg = screen.configure()  # 기존 전체 계획은 seed42~51; 실행 대상만 49로 제한
SEED = 49
MODELS = (screen.M1, screen.M3, screen.M4_C, screen.M5_C)

def locate_existing_run():
    expected_config = json.loads(json.dumps(asdict(cfg)))
    matches = []
    for path in sorted(Path(cfg.out_dir).glob('*/protocol.json')):
        protocol = json.loads(path.read_text())
        if (protocol.get('code_version') == screen.CODE_VERSION
                and protocol.get('config') == expected_config
                and protocol.get('source_revision') == SOURCE_COMMIT):
            matches.append((path, protocol))
    if len(matches) != 1:
        raise RuntimeError(f'일치하는 기존 H&M 실행이 {len(matches)}개입니다. 새 실험을 만들지 않고 중단합니다. 확인 경로: {cfg.out_dir}')
    path, protocol = matches[0]
    return dict(run_dir=path.parent, config_hash=path.parent.name,
                revision=SOURCE_COMMIT, input_hash=protocol['input_hash'], protocol=protocol)

def read_saved(context, model_id):
    row = screen._load_completed_arm(context, model_id, SEED)
    if row is not None:
        metrics = row.get('metrics', {})
        if (row.get('model_id') != model_id or row.get('seed') != SEED
                or not set((*screen.ACCURACY, *screen.ECONOMIC)).issubset(metrics)
                or not all(isinstance(v, (int, float)) and math.isfinite(v) for v in metrics.values())):
            raise RuntimeError(f'저장 지표 신원·값 오류: {model_id}')
    return row

saved = locate_existing_run()
print('기존 실행 폴더:', saved['run_dir'])
for model_id in MODELS:
    row = read_saved(saved, model_id)
    state = '완료 결과 재사용' if row is not None else ('M4-C만 재개/신규 학습' if model_id == screen.M4_C else '미완료 — 이 노트북에서 학습하지 않음')
    print(model_id, ':', state)


In [ ]:
# 유일한 학습 호출: M4-C seed49. 다른 모델의 run_arm/run_batch/run_final은 호출하지 않음.
m4_result = read_saved(saved, screen.M4_C)
if m4_result is not None:
    print('[재사용] M4-C seed49 완료 — 데이터 준비·추가 학습·재평가 없음')
else:
    assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
    assert 'L4' in torch.cuda.get_device_name(0), '런타임 유형을 L4로 변경한 뒤 위부터 실행하세요.'
    print('GPU:', torch.cuda.get_device_name(0))
    # 실행 신원 일치를 위해 기존 준비 함수를 그대로 사용합니다.
    # 그래프/가중치 준비는 포함하지만 M3 또는 M5-C를 학습하지 않습니다.
    if 'prepared' not in globals():
        prepared = screen.prepare(cfg)
    for key in ('run_dir', 'config_hash', 'revision', 'input_hash'):
        assert prepared[key] == saved[key], f'기존 실행과 {key} 불일치 — 학습 중단'
    assert json.loads(json.dumps(prepared['protocol'])) == saved['protocol'], '기존 protocol 불일치'
    assert set(prepared['data']['splits']) == {'test'}
    assert cfg.epochs == 300 and cfg.input_days == 365
    m4_result = screen.run_arm(prepared, cfg, screen.M4_C, SEED)
print('M4-C seed49 완료. 다른 모형의 완료 여부와 무관하게 결과를 저장했습니다.')


In [ ]:
# 비교 모형이 나중에 끝나면 이 셀과 다운로드 셀만 다시 실행합니다(학습·재평가 없음).
import pandas as pd
from IPython.display import display

rows = [row for model_id in MODELS if (row := read_saved(saved, model_id)) is not None]
assert any(row['model_id'] == screen.M4_C for row in rows), 'M4-C 학습 셀을 먼저 완료하세요.'
absolute, diagnostics = screen._tables(rows)
by_model = {row['model_id']: row for row in rows}
pairs = ((screen.M4_C, screen.M1), (screen.M3, screen.M1),
         (screen.M5_C, screen.M1), (screen.M5_C, screen.M3), (screen.M5_C, screen.M4_C))
comparisons = []
for model_id, reference in pairs:
    if model_id not in by_model or reference not in by_model:
        continue
    values, bases = by_model[model_id]['metrics'], by_model[reference]['metrics']
    if set(values) != set(bases):
        raise RuntimeError('동일 실행의 지표 집합 불일치')
    for metric, value in values.items():
        base = bases[metric]
        comparisons.append(dict(model_id=model_id, reference=reference, seed=SEED, metric=metric,
                                value=value, reference_value=base, delta=value-base,
                                relative_change_pct=100*(value/base-1) if base != 0 else None))
comparison = pd.DataFrame(comparisons, columns=['model_id','reference','seed','metric','value','reference_value','delta','relative_change_pct'])
status = dict(seed=SEED, evaluated_epoch=300, m4_complete=True, final_decision_available=False,
              significance_claim=False, retuning_allowed=False,
              missing_references=[m for m in MODELS if m not in by_model],
              note='고정 10시드 계획의 부분 결과. 성공·실패 판정 아님.')
# 다른 GPU의 partial_reports와 파일명이 겹치지 않는 독립 내보내기 폴더
export_dir = saved['run_dir'] / 'm4c_seed49_export'
paths = {}
for name, table in dict(absolute=absolute, comparison=comparison, diagnostics=diagnostics).items():
    path = export_dir / (name + '.csv')
    screen.io._atomic_csv(path, table)
    paths[name] = str(path)
paths['json'] = str(export_dir / 'result.json')
screen.io._atomic_json(Path(paths['json']), dict(protocol=saved['protocol'], status=status,
    source_arm_paths={m: str(screen._arm_path(saved, m, SEED)) for m in by_model}, result_paths=paths))
print(json.dumps(status, ensure_ascii=False, indent=2))
display(absolute[['model_id','seed',*screen.ACCURACY,*screen.ECONOMIC]])
print('전체 지표·세그먼트·진단은 ZIP에 포함됩니다. 비교 모형 미완료는 실패가 아닙니다.')


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

output = Path('/content/clv_hm_m4c_lastweek_seed49_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, Path(path).name)
    archive.write(saved['run_dir'] / 'protocol.json', 'protocol.json')
    for model_id in by_model:
        path = screen._arm_path(saved, model_id, SEED)
        archive.write(path, 'arms/' + path.name)
files.download(str(output))
